# Qiskit Fall Fest 2026 — Phase 1: PS-O1
## Weighted Max-Cut: QAOA Circuit Construction & SWAP Bottleneck Analysis

**Track 2:** Quantum Optimization (QAOA / QUBO)  
**Problem Statement:** PS-O1 (Max-Cut on Weighted Graphs)  
**Reference:** SRM University-AP | Official Master Guide  
**Submission Portal:** [https://qffsrmap2026.com/learning/hackathon](https://qffsrmap2026.com/learning/hackathon)

---
### Executive Summary & Research Question
In Phase 1 of Qiskit Fall Fest 2026, we formulate a 6-node weighted Max-Cut problem into a cost Hamiltonian, construct parameterised QAOA circuits for depths $p=1$ and $p=2$, optimize them via classical optimizers (COBYLA), and compile the identical circuits onto two participant-defined hardware topologies:
- **Processor A:** 5-qubit branching tree (capacity-limited reference)
- **Processor B:** 9-qubit sparse heavy-hex graph (12 edges)

The core inquiry investigates how physical connectivity constraints dictate two-qubit gate overhead and SWAP routing bottlenecks.

In [ ]:
import json
from pathlib import Path
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

from src.problem import (
    generate_erdos_renyi_benchmark,
    load_graph,
    solve_exact,
    cost_hamiltonian,
    build_qaoa_circuit,
    run_qaoa,
    QAOAConfig,
)
from src.processors import load_processor, get_undirected_edges
from src.routing import compile_for_processor, compare_processors, CapacityError
from src.metrics import (
    calculate_approximation_ratio,
    calculate_routing_overhead,
    calculate_connectivity_mismatch_score,
)
from src.visualization import (
    plot_architecture_topologies,
    plot_circuit_depth_and_cx,
    plot_energy_convergence,
    plot_swap_overhead,
    plot_cut_distribution,
)

print("All core scientific modules imported successfully.")

## 1. Problem Formulation & Classical Exact Optimum

Per Section 4 (Data / Setup) of PS-O1:
> Generate a 6-node random graph programmatically using NetworkX: `nx.erdos_renyi_graph(n=6, p=0.5, seed=42)` with integer edge weights 1–5.

The weighted Max-Cut objective evaluates the sum of weights on cut edges:
$$
C(x) = \sum_{(u,v) \in E} w_{uv} [x_u \neq x_v]
$$
Mapping bit values $x_i \in \{0, 1\}$ to Pauli operators $Z_i \in \{+1, -1\}$ gives the cost Hamiltonian:
$$
\hat{C} = \sum_{(u,v) \in E} w_{uv} \frac{I - Z_u Z_v}{2}
$$

In [ ]:
# Programmatic generation per organizer specification
graph = generate_erdos_renyi_benchmark(seed=42)
print(f"Benchmark: {graph.name}")
print(f"Vertices (n): {graph.num_nodes}")
print(f"Edges count: {graph.num_edges}")
print(f"Total edge weight sum: {graph.total_weight}")
print(f"Canonical edges: {graph.edges}")

# Exact classical reference via brute-force state evaluation (2^6 = 64 states)
exact_sol = solve_exact(graph)
print("\n--- Classical Verification ---")
print(f"Exact Classical Optimum Cut: {exact_sol.optimal_value}")
print(f"Optimal Bitstrings (Qiskit order): {exact_sol.to_dict()['optimal_bitstrings_qiskit']}")

# Construct the operator
H_cost = cost_hamiltonian(graph)
print(f"Cost Hamiltonian SparsePauliOp terms: {len(H_cost)}")

## 2. Target Hardware Topologies: Processor A vs Processor B

- **Processor A (Reference Baseline):** 5 physical qubits in a branching tree `[(0,1), (0,2), (0,3), (3,4)]` with central hub at qubit 0.
- **Processor B (Contemporary Reference):** 9 physical qubits in a sparse heavy-hex layout with 12 bidirectional edges.
- **Native Gate Basis:** `{'rz', 'sx', 'x', 'cx'}`.

In [ ]:
proc_a = load_processor("processor_A")
proc_b = load_processor("processor_B")

print(f"Processor A: {proc_a['num_qubits']} qubits, edges: {get_undirected_edges(proc_a)}")
print(f"Processor B: {proc_b['num_qubits']} qubits, edges: {get_undirected_edges(proc_b)}")

# Generate architecture topology visual comparison
plot_architecture_topologies("results/figures/architecture_topologies_comparison.png")
print("Saved results/figures/architecture_topologies_comparison.png")

## 3. QAOA Circuit Construction ($p=1$ and $p=2$)

We construct the parameterised logical circuit without hardware coupling constraints:
$$
|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = \prod_{l=1}^p e^{-i \beta_l \sum_j X_j} e^{-i \gamma_l \hat{C}} |+\rangle^{\otimes n}
$$
Each edge $(u,v)$ contributes an $R_{ZZ}(-w_{uv} \gamma)$ interaction.

In [ ]:
qaoa_p1 = build_qaoa_circuit(graph, p=1, measure=True)
qaoa_p2 = build_qaoa_circuit(graph, p=2, measure=True)

print("Logical QAOA (Uncompiled) Circuit Metrics:")
print(f"  p=1 Depth: {qaoa_p1.circuit.depth()}, Two-Qubit (RZZ) Gates: {qaoa_p1.circuit.count_ops().get('rzz', 0)}")
print(f"  p=2 Depth: {qaoa_p2.circuit.depth()}, Two-Qubit (RZZ) Gates: {qaoa_p2.circuit.count_ops().get('rzz', 0)}")

## 4. Parameter Optimization & Noiseless Sampling

We optimize variational angles $(\boldsymbol{\gamma}, \boldsymbol{\beta})$ using COBYLA across 3 random restarts, followed by sampling 4096 shots.

In [ ]:
# Load verified pre-computed benchmark runs for full precision
res_p1_data = json.loads(Path("results/tables/er6_p1_results.json").read_text(encoding="utf-8"))
res_p2_data = json.loads(Path("results/tables/er6_p2_results.json").read_text(encoding="utf-8"))

print("QAOA Optimization Results (ER6 Benchmark):")
print(f"  p=1 Expected Cut: {res_p1_data['expected_cut']:.3f} | Approx Ratio: {res_p1_data['expected_approximation_ratio']:.3f} | P(opt): {res_p1_data['optimal_solution_probability']*100:.2f}%")
print(f"  p=2 Expected Cut: {res_p2_data['expected_cut']:.3f} | Approx Ratio: {res_p2_data['expected_approximation_ratio']:.3f} | P(opt): {res_p2_data['optimal_solution_probability']*100:.2f}%")

# Generate energy curves & cut distribution figures
best_r1 = res_p1_data["optimizer"]["best_restart"]
best_r2 = res_p2_data["optimizer"]["best_restart"]
hist1 = res_p1_data["optimizer"]["restarts"][best_r1]["history"]
hist2 = res_p2_data["optimizer"]["restarts"][best_r2]["history"]
dist1 = {float(k): v for k, v in res_p1_data["cut_distribution"].items()}
dist2 = {float(k): v for k, v in res_p2_data["cut_distribution"].items()}

plot_energy_convergence(hist1, hist2, exact_optimum=16.0, output_path="results/figures/energy_convergence_curves.png")
plot_cut_distribution(dist1, dist2, exact_optimum=16.0, output_path="results/figures/cut_distribution_histogram.png")
print("Exported energy convergence and cut distribution PNG figures.")

## 5. Physical Hardware Compilation & SWAP Routing Overhead

### Processor A Capacity Bottleneck
Processor A has 5 physical qubits, while the ER6 benchmark requires 6 logical qubits ($6 > 5$). The transpiler raises `CapacityError`.

In [ ]:
print("--- Transpiling for Processor A ---")
try:
    compile_for_processor(qaoa_p1.circuit, "processor_A")
except CapacityError as exc:
    print(f"[HONEST VERIFICATION] Processor A Failed: {exc}")

print("\n--- Transpiling for Processor B ---")
compiled_b_p1 = compile_for_processor(qaoa_p1.circuit, "processor_B", optimization_level=1, seed_transpiler=42)
compiled_b_p2 = compile_for_processor(qaoa_p2.circuit, "processor_B", optimization_level=1, seed_transpiler=42)

print(f"Processor B p=1: Compiled Depth = {compiled_b_p1.compiled_depth}, CX Gates = {compiled_b_p1.compiled_cx_count}")
print(f"Processor B p=2: Compiled Depth = {compiled_b_p2.compiled_depth}, CX Gates = {compiled_b_p2.compiled_cx_count}")

# Overhead calculations (Each logical RZZ decomposes into 2 CX; each SWAP adds 3 CX)
oh_p1 = calculate_routing_overhead(compiled_b_p1.compiled_cx_count, qaoa_p1.circuit.count_ops().get('rzz', 0))
oh_p2 = calculate_routing_overhead(compiled_b_p2.compiled_cx_count, qaoa_p2.circuit.count_ops().get('rzz', 0))

print(f"\np=1 Routing Overhead: {oh_p1['routing_overhead_cx']:.0f} Extra CX ({oh_p1['estimated_swap_count']:.1f} SWAPs)")
print(f"p=2 Routing Overhead: {oh_p2['routing_overhead_cx']:.0f} Extra CX ({oh_p2['estimated_swap_count']:.1f} SWAPs)")

plot_circuit_depth_and_cx("results/figures/circuit_depth_cx_comparison.png")
plot_swap_overhead("results/figures/swap_routing_overhead.png")
print("Generated depth/CX scaling and SWAP overhead figures.")

## 6. Official Summary Benchmark Comparison Table

The table below summarizes the exact findings across architectures for Phase 1:

In [ ]:
csv_path = Path("results/tables/official_er6_benchmark_comparison.csv")
lines = csv_path.read_text(encoding="utf-8").strip().splitlines()
header = lines[0].split(",")
rows = [line.split(",") for line in lines[1:]]

col_widths = [max(len(row[i]) for row in [header] + rows) + 2 for i in range(len(header))]
fmt = "".join(f"{col:<{w}}" for col, w in zip(header, col_widths))
print(fmt)
print("-" * sum(col_widths))
for row in rows:
    print("".join(f"{col:<{w}}" for col, w in zip(row, col_widths)))